# RNN Project

Twitter Analysis

1. Data Gathering

In [1]:
import pandas as pd

In [2]:
data=pd.read_csv('twitter_training.csv')

In [3]:
data.shape

(74681, 4)

In [4]:
data=data.sample(15000, random_state=42)

In [5]:
data.shape

(15000, 4)

2. Data Exploration

In [6]:
data['Positive'].value_counts()

Positive
Negative      4488
Positive      4275
Neutral       3568
Irrelevant    2669
Name: count, dtype: int64

In [7]:
data.isnull().sum()

2401                                                       0
Borderlands                                                0
Positive                                                   0
im getting on borderlands and i will murder you all ,    151
dtype: int64

In [8]:
data.duplicated().sum()

np.int64(127)

In [9]:
data.info()

<class 'pandas.DataFrame'>
Index: 15000 entries, 34877 to 56434
Data columns (total 4 columns):
 #   Column                                                 Non-Null Count  Dtype
---  ------                                                 --------------  -----
 0   2401                                                   15000 non-null  int64
 1   Borderlands                                            15000 non-null  str  
 2   Positive                                               15000 non-null  str  
 3   im getting on borderlands and i will murder you all ,  14849 non-null  str  
dtypes: int64(1), str(3)
memory usage: 585.9 KB


2. Data Preprocessing & Cleaning

In [10]:
data=data.drop_duplicates()

In [11]:
data.shape

(14873, 4)

In [12]:
data=data.dropna()

In [13]:
data.isnull().sum()

2401                                                     0
Borderlands                                              0
Positive                                                 0
im getting on borderlands and i will murder you all ,    0
dtype: int64

In [14]:
data.duplicated().sum()

np.int64(0)

In [15]:
data.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
34877,6792,Fortnite,Irrelevant,went to go in george's room to find his door w...
21704,4115,CS-GO,Positive,Yo this looks LIT! Team:GO/Overwatch combo
47008,5665,HomeDepot,Negative,Pay attention executive administrators. While ...
7969,9369,Overwatch,Irrelevant,Guy looked at me and says my name was put on t...
454,2476,Borderlands,Positive,one


In [16]:
data=data.drop(columns=['2401','Borderlands'])

In [17]:
data.head()

,Positive,"im getting on borderlands and i will murder you all ,"
34877,Irrelevant,went to go in george's room to find his door w...
21704,Positive,Yo this looks LIT! Team:GO/Overwatch combo
47008,Negative,Pay attention executive administrators. While ...
7969,Irrelevant,Guy looked at me and says my name was put on t...
454,Positive,one


In [18]:
data=data.rename(columns={'Positive':'Sentiment','im getting on borderlands and i will murder you all ,':'Tweets'})

In [19]:
data.head()

,Sentiment,Tweets
34877,Irrelevant,went to go in george's room to find his door w...
21704,Positive,Yo this looks LIT! Team:GO/Overwatch combo
47008,Negative,Pay attention executive administrators. While ...
7969,Irrelevant,Guy looked at me and says my name was put on t...
454,Positive,one


In [20]:
data=data[data['Sentiment']!='Irrelevant']

In [21]:
data.shape

(12127, 2)

In [22]:
data['Sentiment'].value_counts()

Sentiment
Negative    4418
Positive    4201
Neutral     3508
Name: count, dtype: int64

3. Feature Engineering

In [23]:
from sklearn.preprocessing import LabelEncoder

In [24]:
le=LabelEncoder()

In [25]:
data['Sentiment']=le.fit_transform(data['Sentiment'])

In [26]:
data['Sentiment'].head(10)

21704    2
47008    0
454      2
58076    0
40659    2
3235     2
46582    1
44437    2
13037    0
7299     2
Name: Sentiment, dtype: int64

In [27]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [28]:
token=Tokenizer(oov_token='Nothing')

In [29]:
data['Tweets'].head()

21704           Yo this looks LIT! Team:GO/Overwatch combo
47008    Pay attention executive administrators. While ...
454                                                    one
58076         @Rainbow6Game Server are Available in Xbox 🥺
40659    so Awesome siege attempt... strangest bleeding...
Name: Tweets, dtype: str

In [30]:
token.fit_on_texts(data['Tweets'])

In [31]:
data['Tweets'].head()

21704           Yo this looks LIT! Team:GO/Overwatch combo
47008    Pay attention executive administrators. While ...
454                                                    one
58076         @Rainbow6Game Server are Available in Xbox 🥺
40659    so Awesome siege attempt... strangest bleeding...
Name: Tweets, dtype: str

In [32]:
token.document_count

12127

In [33]:
len(token.word_index) ### Vocabulary Size

17872

In [34]:
sequences=token.texts_to_sequences(data['Tweets'])

In [35]:
data['Tweets'].isnull().sum()

np.int64(0)

In [36]:
from keras.utils import pad_sequences

In [37]:
sequences=pad_sequences(sequences,padding='pre')

In [38]:
sequences

array([[    0,     0,     0, ...,    95,   197,  2525],
       [    0,     0,     0, ...,     7,  2163,  3098],
       [    0,     0,     0, ...,     0,     0,    52],
       ...,
       [    0,     0,     0, ...,    20,   223,  6168],
       [    0,     0,     0, ...,    43,    74, 17872],
       [    0,     0,     0, ...,   792,  1031,  6492]],
      shape=(12127, 99), dtype=int32)

In [40]:
from tensorflow.keras.layers import SimpleRNN, Dense, Input, Embedding
from tensorflow.keras.models import Sequential

In [41]:
X=sequences

In [42]:
y=data['Sentiment'].values

In [43]:
from sklearn.model_selection import train_test_split

In [44]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [45]:
X_train.shape

(9701, 99)

In [46]:
# Training

In [47]:
model=Sequential()
model.add(SimpleRNN(32,input_shape=(99,1),return_sequences=False))
model.add(Dense(3,activation='softmax'))
model.summary()

c:\Users\rosha\anaconda3\envs\tf_env\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 32)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,187 (4.64 KB)

 Trainable params: 1,187 (4.64 KB)

 Non-trainable params: 0 (0.00 B)

In [48]:
model.compile(optimizer='adam',metrics=['accuracy'],loss='sparse_categorical_crossentropy')

In [49]:
model.fit(X_train,y_train,epochs=10,validation_data=(X_test,y_test))

Epoch 1/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - accuracy: 0.3620 - loss: 1.1018 - val_accuracy: 0.3475 - val_loss: 1.1021
Epoch 2/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.3590 - loss: 1.0944 - val_accuracy: 0.3405 - val_loss: 1.0970
Epoch 3/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.3570 - loss: 1.0942 - val_accuracy: 0.3071 - val_loss: 1.1054
Epoch 4/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.3619 - loss: 1.0932 - val_accuracy: 0.3594 - val_loss: 1.0973
Epoch 5/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.3610 - loss: 1.0907 - val_accuracy: 0.3495 - val_loss: 1.0972
Epoch 6/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.3609 - loss: 1.0925 - val_accuracy: 0.3689 - val_loss: 1.0952
Epoch 7/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.3706 - loss: 1.0910 - val_accuracy: 0.3467 - val_loss: 1.0949
Epoch 8/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - accuracy: 0.3613 - loss: 1.0918 - val_acc

Above Model is trained without using embedding layer

Now, with Embedding Layer Inserted

In [50]:
vocabulary_size=len(token.word_index)+1

In [51]:
vocabulary_size

17873

In [52]:
model_2=Sequential()

In [53]:
model_2.add(Input(shape=(99,)))
model_2.add(Embedding(input_dim=vocabulary_size,output_dim=64))
model_2.add(Dense(3,activation='softmax'))

In [54]:
model_2.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 99, 64)         │     1,143,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 99, 3)          │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,144,067 (4.36 MB)

 Trainable params: 1,144,067 (4.36 MB)

 Non-trainable params: 0 (0.00 B)

In [55]:
model_final=Sequential([Embedding(input_dim=vocabulary_size,output_dim=128,input_length=99,mask_zero=True),SimpleRNN(64),Dense(3,activation='softmax')])

c:\Users\rosha\anaconda3\envs\tf_env\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [56]:
model_final.compile(optimizer='adam',metrics=['accuracy'],loss='sparse_categorical_crossentropy')

In [57]:
model_final.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [58]:
model_final.fit(X_train,y_train,epochs=10,validation_data=(X_test,y_test))

Epoch 1/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 38s 101ms/step - accuracy: 0.5668 - loss: 0.9105 - val_accuracy: 0.6937 - val_loss: 0.7404
Epoch 2/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 29s 96ms/step - accuracy: 0.8886 - loss: 0.3186 - val_accuracy: 0.7049 - val_loss: 0.7733
Epoch 3/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 30s 97ms/step - accuracy: 0.9665 - loss: 0.0974 - val_accuracy: 0.6826 - val_loss: 0.9685
Epoch 4/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 31s 101ms/step - accuracy: 0.9760 - loss: 0.0611 - val_accuracy: 0.7057 - val_loss: 1.0016
Epoch 5/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 34s 113ms/step - accuracy: 0.9789 - loss: 0.0486 - val_accuracy: 0.7090 - val_loss: 1.0132
Epoch 6/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 34s 111ms/step - accuracy: 0.9805 - loss: 0.0426 - val_accuracy: 0.7234 - val_loss: 1.0669
Epoch 7/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 38s 126ms/step - accuracy: 0.9813 - loss: 0.0418 - val_accuracy: 0.7090 - val_loss: 1.1086
Epoch 8/10
304/304 ━━━━━━━━━━━━━━━━━━━━ 28s 92ms/step - accuracy: 0.9803 - loss: 0.03

In [59]:
import pickle


In [60]:
with open('tokenizer.pkl','wb') as file:
    pickle.dump(token,file)


In [61]:
model_final.save('model.h5')